In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

In [2]:
df_liquidation = pd.read_csv(
    '../data/processed/aave_liquidation_processed.csv',
    parse_dates=['block_timestamp', 'date', 'minute']
)

df_aggregator = pd.read_csv(
    '../data/processed/chainlink_aggregator_processed.csv',
    parse_dates=['block_timestamp', 'date', 'minute', 'updated_at_datetime']
)

df_aave_tvl = pd.read_csv('../data/raw/AAVE_Tvl_Data.csv')
df_aave_tvl['Date'] = pd.to_datetime(df_aave_tvl['Date'], format='mixed')

df_eth_tvl = pd.read_csv(
    '../data/raw/ETH_Total_Tvl_Data.csv',
    parse_dates=['Date']
)


In [3]:
oracle_minute = (
    df_aggregator
    .dropna(subset=["minute"])
    .sort_values(["feed_name", "minute", "block_timestamp"])
    .groupby(["minute", "feed_name", "symbol"])
    .agg(
        oracle_updates=("transaction_hash", "count"),
        mean_oracle_price=("oracle_price", "mean"),
        last_oracle_price=("oracle_price", "last"),
        mean_benchmark_price=("benchmark_close", "mean"),
        last_benchmark_price=("benchmark_close", "last"),
        mean_oracle_error=("oracle_error", "mean"),
        max_abs_oracle_error=("abs_oracle_error", "max"),
        mean_abs_oracle_error=("abs_oracle_error", "mean"),
        shock_05pct=("shock_05pct", "max"),
        shock_1pct=("shock_1pct", "max"),
        shock_2pct=("shock_2pct", "max"),
        shock_5pct=("shock_5pct", "max")
    )
    .reset_index()
)

oracle_minute["max_abs_oracle_error_pct"] = oracle_minute["max_abs_oracle_error"] * 100
oracle_minute["mean_abs_oracle_error_pct"] = oracle_minute["mean_abs_oracle_error"] * 100

oracle_system_minute = (
    oracle_minute
    .groupby("minute")
    .agg(
        oracle_updates=("oracle_updates", "sum"),
        max_abs_oracle_error=("max_abs_oracle_error", "max"),
        mean_abs_oracle_error=("mean_abs_oracle_error", "mean"),
        shock_05pct=("shock_05pct", "max"),
        shock_1pct=("shock_1pct", "max"),
        shock_2pct=("shock_2pct", "max"),
        shock_5pct=("shock_5pct", "max")
    )
    .reset_index()
)

oracle_system_minute["max_abs_oracle_error_pct"] = oracle_system_minute["max_abs_oracle_error"] * 100
oracle_system_minute["mean_abs_oracle_error_pct"] = oracle_system_minute["mean_abs_oracle_error"] * 100

# Feed responsible for maximum deviation in each minute
idx_max = (
    oracle_minute
    .sort_values(["minute", "max_abs_oracle_error"], ascending=[True, False])
    .groupby("minute")
    .head(1)
    [["minute", "feed_name", "symbol", "max_abs_oracle_error_pct"]]
    .rename(
        columns={
            "feed_name": "max_error_feed",
            "symbol": "max_error_symbol",
            "max_abs_oracle_error_pct": "feed_level_max_abs_oracle_error_pct"
        }
    )
)

oracle_system_minute = oracle_system_minute.merge(
    idx_max,
    on="minute",
    how="left"
)

oracle_system_minute.head()

,minute,oracle_updates,max_abs_oracle_error,mean_abs_oracle_error,shock_05pct,shock_1pct,shock_2pct,shock_5pct,max_abs_oracle_error_pct,mean_abs_oracle_error_pct,max_error_feed,max_error_symbol,feed_level_max_abs_oracle_error_pct
0,2021-01-01 00:04:00+00:00,10,0.003244,0.002571,0,0,0,0,0.324430,0.257129,ETHUSD,ETHUSDT,0.324430
1,2021-01-01 00:05:00+00:00,7,0.004453,0.003613,0,0,0,0,0.445303,0.361297,ETHUSD,ETHUSDT,0.445303
2,2021-01-01 00:06:00+00:00,2,0.003771,0.003341,0,0,0,0,0.377088,0.334079,ETHUSD,ETHUSDT,0.377088
3,2021-01-01 00:08:00+00:00,8,0.006232,0.006087,1,0,0,0,0.623221,0.608651,LINKUSD,LINKUSDT,0.623221
4,2021-01-01 00:12:00+00:00,16,0.003462,0.003022,0,0,0,0,0.346198,0.302162,BTCUSD,BTCUSDT,0.346198


In [4]:
liq_total_by_minute = (
    df_liquidation
    .groupby("minute")
    .agg(
        liquidation_count=("transaction_hash", "count"),
        unique_borrowers=("borrower", "nunique"),
        unique_liquidators=("liquidator", "nunique")
    )
    .reset_index()
)

liq_total_by_minute.head()

,minute,liquidation_count,unique_borrowers,unique_liquidators
0,2020-12-17 14:19:00+00:00,1,1,1
1,2020-12-17 14:21:00+00:00,2,1,2
2,2020-12-17 14:23:00+00:00,1,1,1
3,2020-12-17 15:20:00+00:00,1,1,1
4,2020-12-18 00:34:00+00:00,1,1,1


In [5]:
aave_tvl_daily = (
    df_aave_tvl
    .dropna(subset=["Date", "TVL_USD"])
    .groupby("Date")
    .agg(aave_tvl_usd=("TVL_USD", "sum"))
    .reset_index()
)

aave_tvl_daily.head()

,Date,aave_tvl_usd
0,2020-12-03,1.133852e+04
1,2020-12-04,2.920358e+06
2,2020-12-05,1.492493e+07
3,2020-12-06,1.853665e+07
4,2020-12-07,1.987649e+07


In [6]:
eth_tvl_daily = (
    df_eth_tvl
    .dropna(subset=["Date", "Ethereum_TVL_USD"])
    .groupby("Date")
    .agg(ethereum_tvl_usd=("Ethereum_TVL_USD", "mean"))
    .reset_index()
)

eth_tvl_daily.head()

,Date,ethereum_tvl_usd
0,2017-09-27,0.0
1,2017-09-28,0.0
2,2017-09-29,0.0
3,2017-09-30,0.0
4,2017-10-01,0.0


In [7]:
start_time = pd.Timestamp("2021-01-01 00:00:00", tz="UTC")
end_time = pd.Timestamp("2024-12-31 23:59:00", tz="UTC")

panel_a = pd.DataFrame({
    "minute": pd.date_range(start=start_time, end=end_time, freq="min")
})

panel_a["Date"] = panel_a["minute"].dt.floor("D").dt.tz_localize(None)

# MERGE LIQUIDATION
panel_a = panel_a.merge(
    liq_total_by_minute,
    on="minute",
    how="left"
)

for col in ["liquidation_count", "unique_borrowers", "unique_liquidators"]:
    panel_a[col] = panel_a[col].fillna(0).astype(int)

# MERGE ORACLE
panel_a = panel_a.merge(
    oracle_system_minute,
    on="minute",
    how="left"
)

zero_cols = [
    "oracle_updates",
    "max_abs_oracle_error",
    "mean_abs_oracle_error",
    "shock_05pct",
    "shock_1pct",
    "shock_2pct",
    "shock_5pct",
    "max_abs_oracle_error_pct",
    "mean_abs_oracle_error_pct"
]

for col in zero_cols:
    panel_a[col] = panel_a[col].fillna(0)

for col in ["max_error_feed", "max_error_symbol"]:
    panel_a[col] = panel_a[col].fillna("none")

# CALCULATE FUTURE SUM
def future_sum(series, horizon):
    """
    Sum from t+1 to t+horizon.
    Current minute t is excluded.
    """
    return (
        series
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .sum()
        .iloc[::-1]
    )

for horizon in [5, 15, 30, 60, 120]:
    panel_a[f"liq_next_{horizon}min"] = future_sum(panel_a["liquidation_count"], horizon)
    panel_a[f"borrowers_next_{horizon}min"] = future_sum(panel_a["unique_borrowers"], horizon)
    panel_a[f"liquidators_next_{horizon}min"] = future_sum(panel_a["unique_liquidators"], horizon)


panel_a["Date"] = panel_a["minute"].dt.floor("D").dt.tz_localize(None)

panel_a = panel_a.merge(aave_tvl_daily, on="Date", how="left")
panel_a = panel_a.merge(eth_tvl_daily, on="Date", how="left")

panel_a["aave_tvl_usd"] = panel_a["aave_tvl_usd"].ffill().bfill()
panel_a["ethereum_tvl_usd"] = panel_a["ethereum_tvl_usd"].ffill().bfill()

# Normalized stress
panel_a["liq_count_per_billion_aave_tvl"] = (
    panel_a["liquidation_count"] / (panel_a["aave_tvl_usd"] / 1e9)
)

for horizon in [5, 15, 30, 60, 120]:
    panel_a[f"liq_next_{horizon}min_per_billion_aave_tvl"] = (
        panel_a[f"liq_next_{horizon}min"] / (panel_a["aave_tvl_usd"] / 1e9)
    )

panel_a["hour"] = panel_a["minute"].dt.hour
panel_a["day_of_week"] = panel_a["minute"].dt.dayofweek
panel_a["date"] = panel_a["minute"].dt.date
panel_a["month"] = panel_a["minute"].dt.strftime("%Y-%m")

panel_a["log_aave_tvl"] = np.log(panel_a["aave_tvl_usd"])
panel_a["log_ethereum_tvl"] = np.log(panel_a["ethereum_tvl_usd"])

panel_a = panel_a.replace([np.inf, -np.inf], np.nan)

panel_a.head()



,minute,Date,liquidation_count,unique_borrowers,unique_liquidators,oracle_updates,max_abs_oracle_error,mean_abs_oracle_error,shock_05pct,shock_1pct,shock_2pct,shock_5pct,max_abs_oracle_error_pct,mean_abs_oracle_error_pct,max_error_feed,max_error_symbol,feed_level_max_abs_oracle_error_pct,liq_next_5min,borrowers_next_5min,liquidators_next_5min,liq_next_15min,borrowers_next_15min,liquidators_next_15min,liq_next_30min,borrowers_next_30min,liquidators_next_30min,liq_next_60min,borrowers_next_60min,liquidators_next_60min,liq_next_120min,borrowers_next_120min,liquidators_next_120min,aave_tvl_usd,ethereum_tvl_usd,liq_count_per_billion_aave_tvl,liq_next_5min_per_billion_aave_tvl,liq_next_15min_per_billion_aave_tvl,liq_next_30min_per_billion_aave_tvl,liq_next_60min_per_billion_aave_tvl,liq_next_120min_per_billion_aave_tvl,hour,day_of_week,date,month,log_aave_tvl,log_ethereum_tvl
0,2021-01-01 00:00:00+00:00,2021-01-01,0,0,0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,none,none,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.472021e+08,1.471852e+10,0.0,0.0,0.0,0.0,0.0,0.0,0,4,2021-01-01,2021-01,18.807317,23.412372
1,2021-01-01 00:01:00+00:00,2021-01-01,0,0,0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,none,none,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.472021e+08,1.471852e+10,0.0,0.0,0.0,0.0,0.0,0.0,0,4,2021-01-01,2021-01,18.807317,23.412372
2,2021-01-01 00:02:00+00:00,2021-01-01,0,0,0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,none,none,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.472021e+08,1.471852e+10,0.0,0.0,0.0,0.0,0.0,0.0,0,4,2021-01-01,2021-01,18.807317,23.412372
3,2021-01-01 00:03:00+00:00,2021-01-01,0,0,0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,none,none,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.472021e+08,1.471852e+10,0.0,0.0,0.0,0.0,0.0,0.0,0,4,2021-01-01,2021-01,18.807317,23.412372
4,2021-01-01 00:04:00+00:00,2021-01-01,0,0,0,10.0,0.003244,0.002571,0.0,0.0,0.0,0.0,0.32443,0.257129,ETHUSD,ETHUSDT,0.32443,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.472021e+08,1.471852e+10,0.0,0.0,0.0,0.0,0.0,0.0,0,4,2021-01-01,2021-01,18.807317,23.412372


In [8]:
panel_a.to_csv('../data/final/Model_A_Reconstructed_Minute_Panel.csv', index=False)